<a href="https://colab.research.google.com/github/Andrian0s/ML4NLP1-2026-Tutorial-Notebooks/blob/main/tutorials_notebooks_in_class_2026/W10_introduction_to_prompt_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Prompting base and instruction-tuned LLMs**

In this notebook, we give the same prompts to a base LLM and to its instruction-tuned version and look at what they do. We use `Llama-3.2-1B` and `Llama-3.2-1B-Instruct`. They belong to the same family and use the same tokenizer as the 3B models in Exercise 5, but they are small enough to run quickly.

We follow four questions:

1. What does a base model do with a question?
2. How do few-shot examples change its output?
3. When does generation stop?
4. What changes with the instruction-tuned model and its chat template?

A GPU makes the notebook faster (in Colab: "Runtime" → "Change runtime type" → T4 GPU), but it also runs on a CPU.

In [1]:
%pip install -q transformers==5.16.1

import gc
import torch
from transformers import pipeline
from transformers.utils import logging

logging.set_verbosity_error() # hide warnings from transformers that do not matter here

# Pick the best available device: NVIDIA GPU, then Apple Silicon GPU, then CPU
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
# Half precision on a GPU, full precision on a CPU
dtype = torch.float16 if device != 'cpu' else torch.float32
print(device)

Note: you may need to restart the kernel to use updated packages.


mps


## 1) What does a base model do with a question?

A base model was trained on one task only: predict the next token of a text. We load it with a `text-generation` pipeline. The models come from the `unsloth` organisation on the Hugging Face Hub, which hosts copies of Meta's Llama 3.2 models that you can download without requesting access first (the Llama 3.2 licence still applies).

In [2]:
base = pipeline("text-generation", model="unsloth/Llama-3.2-1B", device=device, dtype=dtype)

We use greedy decoding (`do_sample=False`): at every step, the model takes the most probable next token, so the output is the same every time we run the cell. `return_full_text=False` returns only the new text, without the prompt.

In [3]:
def generate(pipe, prompt, max_new_tokens=50):
    """Generates greedily and returns only the new text."""
    output = pipe(prompt, max_new_tokens=max_new_tokens, do_sample=False, return_full_text=False)
    return output[0]["generated_text"]

print(generate(base, "What is the capital of Switzerland?"))

?
What is the time zone in Switzerland?
What is the currency in Switzerland?
What is the population of Switzerland?
What is the largest city in Switzerland?
What is the capital of Switzerland?
What is the time zone in Switzerland?
What is the currency


The model does not answer. It treats the question as the beginning of a list of questions, the kind of text it has seen on the web (FAQ pages, quizzes), and continues the list: the time zone, the currency, the population. It even repeats the first question and starts over. It stops only because it reaches `max_new_tokens`.

In [4]:
print(generate(base, "The capital of Switzerland is", max_new_tokens=20))

 a city of contrasts. The city is a mix of old and new, with a rich history and


Closer, but still not what we wanted: "a city of contrasts" is a plausible way to continue the sentence, and the model never says "Bern". The most probable continuation is not necessarily the fact we are after. We need to show the model what kind of continuation we expect.

## 2) How do few-shot examples change its output?

We take a small task: give the opposite of a word. First, we only describe the task (zero-shot).

In [5]:
zero_shot = """Give the opposite of the word.

Word: early
Opposite:"""

print(generate(base, zero_shot))

 late
Example: I will be late for work.
Word: late
Opposite: early
Example: I will be early for work.
Word: early
Opposite: late
Example: I will be late for work.
Word: late


The first word, "late", is right. But then the model keeps writing: an example sentence, then new "Word: … Opposite: …" pairs that it makes up itself.

In [6]:
few_shot = """Give the opposite of the word.

Word: hot
Opposite: cold

Word: big
Opposite: small

Word: fast
Opposite: slow

Word: early
Opposite:"""

few_shot_output = generate(base, few_shot)
print(few_shot_output)

 late

Word: small
Opposite: big

Word: fast
Opposite: slow

Word: early
Opposite: late

Word: big
Opposite: small

Word: fast
Opposite: slow

Word: early


With three examples, the format is clear: the answer comes right after "Opposite:", and the model copies the pattern exactly. After the answer, it continues the pattern with more pairs, partly copied from the examples. With a base model, we therefore cut the output ourselves, here after the first line.

In [7]:
answer = few_shot_output.strip().split("\n")[0]
print(answer)

late


## 3) When does generation stop?

Before the model sees a prompt, the tokenizer turns it into tokens. The Llama tokenizer also adds a special token at the start: the beginning-of-sequence (BOS) token `<|begin_of_text|>`. The `Ġ` at the start of a token stands for a space.

In [8]:
tokenizer = base.tokenizer

print(tokenizer.convert_ids_to_tokens(tokenizer("The capital of Switzerland is")["input_ids"]))
print("BOS token:", tokenizer.bos_token, "| EOS token:", tokenizer.eos_token)

['<|begin_of_text|>', 'The', 'Ġcapital', 'Ġof', 'ĠSwitzerland', 'Ġis']
BOS token: <|begin_of_text|> | EOS token: <|end_of_text|>


The tokenizer adds the BOS token to every text, even if the text already starts with it. In that case, the model sees it twice. With `add_special_tokens=False`, the tokenizer adds nothing.

In [9]:
text = tokenizer.bos_token + "The capital of Switzerland is"

print(tokenizer.convert_ids_to_tokens(tokenizer(text)["input_ids"])[:3])
print(tokenizer.convert_ids_to_tokens(tokenizer(text, add_special_tokens=False)["input_ids"])[:3])

['<|begin_of_text|>', '<|begin_of_text|>', 'The']
['<|begin_of_text|>', 'The', 'Ġcapital']


The end-of-sequence (EOS) token works the other way round: the tokenizer never adds it, the model has to generate it. Generation stops as soon as the model generates one of its EOS tokens, or when `max_new_tokens` is reached. The model's generation config lists its EOS tokens.

In [10]:
eos_ids = base.model.generation_config.eos_token_id
print(eos_ids, tokenizer.convert_ids_to_tokens(eos_ids if isinstance(eos_ids, list) else [eos_ids]))

def count_tokens(pipe, text):
    """Number of tokens in a generated text."""
    return len(pipe.tokenizer(text, add_special_tokens=False)["input_ids"])

print("Tokens in the few-shot output:", count_tokens(base, few_shot_output), "of max_new_tokens = 50")

128001 ['<|end_of_text|>']
Tokens in the few-shot output: 50 of max_new_tokens = 50


The base model used all 50 new tokens and never generated `<|end_of_text|>`: it stopped only because of `max_new_tokens`.

A side note on decoding: the default generation config of these models samples (`do_sample=True`, `temperature=0.6`), so without `do_sample=False`, every run can give a different output.

In [11]:
print(base.model.generation_config)

for seed in [1, 2]:
    torch.manual_seed(seed)
    output = base(few_shot, max_new_tokens=12, do_sample=True, return_full_text=False)[0]["generated_text"]
    print(f"seed {seed}: {output!r}")

GenerationConfig {
  "bos_token_id": 128000,
  "do_sample": true,
  "eos_token_id": 128001,
  "max_length": 131072,
  "pad_token_id": 128004,
  "temperature": 0.6,
  "top_p": 0.9
}

seed 1: ' late\n\nWord: early\nOpposite: late\n\nWord'
seed 2: ' late\n\nWord: fast\nOpposite: slow\n\nWord'


We are done with the base model and free its memory before we load the next model.

In [12]:
del base, tokenizer
gc.collect()
if device == 'cuda':
    torch.cuda.empty_cache()
elif device == 'mps':
    torch.mps.empty_cache()

## 4) What changes with the instruction-tuned model and its chat template?

The instruction-tuned model was trained further on conversations, always written in one fixed format: its chat template. The template is stored with the tokenizer as a small Jinja program. Look at its first line.

In [18]:
instruct = pipeline("text-generation", model="unsloth/Llama-3.2-1B-Instruct", device=device, dtype=dtype)

print(instruct.tokenizer.chat_template)

{{- bos_token }}
{%- if custom_tools is defined %}
    {%- set tools = custom_tools %}
{%- endif %}
{%- if not tools_in_user_message is defined %}
    {%- set tools_in_user_message = true %}
{%- endif %}
{%- if not date_string is defined %}
    {%- if strftime_now is defined %}
        {%- set date_string = strftime_now("%d %b %Y") %}
    {%- else %}
        {%- set date_string = "26 Jul 2024" %}
    {%- endif %}
{%- endif %}
{%- if not tools is defined %}
    {%- set tools = none %}
{%- endif %}

{#- This block extracts the system message, so we can slot it into the right place. #}
{%- if messages[0]['role'] == 'system' %}
    {%- set system_message = messages[0]['content']|trim %}
    {%- set messages = messages[1:] %}
{%- else %}
    {%- set system_message = "" %}
{%- endif %}

{#- System message #}
{{- "<|start_header_id|>system<|end_header_id|>\n\n" }}
{%- if tools is not none %}
    {{- "Environment: ipython\n" }}
{%- endif %}
{{- "Cutting Knowledge Date: December 2023\n" }}
{{- 

The first thing the template writes is `bos_token`. A prompt built with the chat template therefore already starts with `<|begin_of_text|>`. When we pass a list of messages, the pipeline applies the chat template for us and does not add a second BOS token.

In [14]:
messages = [{"role": "user", "content": "What is the capital of Switzerland?"}]

chat_output = generate(instruct, messages)
print(chat_output)
print("Tokens:", count_tokens(instruct, chat_output), "of max_new_tokens = 50")

The capital of Switzerland is Bern.
Tokens: 7 of max_new_tokens = 50


The instruction-tuned model answers in one sentence and stops after 7 tokens, far below the limit.

In [15]:
eos_ids = instruct.model.generation_config.eos_token_id
print(eos_ids, instruct.tokenizer.convert_ids_to_tokens(eos_ids))

[128001, 128008, 128009] ['<|end_of_text|>', '<|eom_id|>', '<|eot_id|>']


The instruction-tuned model has three EOS tokens. `<|eot_id|>` ("end of turn") closes every message in the chat template, and the model generates it at the end of its answer.

In [16]:
print(generate(instruct, "What is the capital of Switzerland?"))

 Bern?
No, that is not correct. The capital of Switzerland is Bern. However, the capital of the canton of Bern is Bern.

The capital of Switzerland is Bern. Bern is a city located in the central part of the country, and


Without the chat template, the same model gets plain text, a format it was not tuned for. It continues "What is the capital of Switzerland?" as a text, argues with itself and runs until `max_new_tokens`. Instruction-tuned models need their chat template.

In [17]:
messages = [
    {"role": "system", "content": "You are a helpful assistant. Always answer in German."},
    {"role": "user", "content": "What is the capital of Switzerland?"},
]

print(generate(instruct, messages))

Die Hauptstadt der Schweiz ist Bern.


A system message sets how the model should behave for the whole conversation. Here, it changes the language of the answer. In practice, system prompts are often long and fixed, while user messages carry the actual requests.

## Summary

- A base model continues text. Frame the task as the beginning of a text, show the format with a few examples, and cut the output after the answer.
- An instruction-tuned model expects its chat template. The template adds the special tokens, including `<|begin_of_text|>`, so the tokenizer must not add it a second time.
- Generation stops at one of the model's EOS tokens or at `max_new_tokens`. Set `do_sample=False` to get the same output every time.